In [ ]:
# ============================================================
# 导入所需的库
# ============================================================
import os
import requests                    # 用来向后端发 HTTP 请求
from PIL import Image              # 处理图片
from io import BytesIO             # 把图片数据转成可读格式
import streamlit as st             # 网页界面框架

# ============================================================
# 读取环境变量
# 后端地址：在 Docker 里是 http://app_backend:8001
# 本地测试时是 http://localhost:8001
# ============================================================
BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8001")

# ============================================================
# 页面基本设置（必须是第一个 streamlit 命令）
# ============================================================
st.set_page_config(
    page_title="NASA Space IR",
    page_icon="🔭",
    layout="wide"
)

# ============================================================
# 页面标题
# ============================================================
st.title("🔭 NASA Deep Space Image Search")
st.caption("Search NASA's deep space image archive by scientific concept or visual appearance.")

# ============================================================
# 左边侧边栏：搜索设置
# ============================================================
with st.sidebar:
    st.header("Search Settings")

    # 选择搜索模式
    search_focus = st.radio(
        "Search Focus",
        options=[
            "Scientific Context (BERT)",   # 文字语义搜索
            "Visual Features (CLIP)"        # 视觉特征搜索
        ]
    )

    # 返回几条结果
    top_k = st.slider(
        "Images to Retrieve",
        min_value=1,
        max_value=10,
        value=5
    )

    # 是否开启 AI 总结
    enable_rag = st.toggle("Enable AI Image Explainer", value=False)

    st.divider()
    st.caption("Powered by BERT · CLIP · Llama 3 · ChromaDB")

# ============================================================
# 根据用户选择，决定搜索模式
# ============================================================
if "Scientific" in search_focus:
    current_search_mode = "text"
else:
    current_search_mode = "image"

# ============================================================
# 搜索框
# ============================================================
query = st.text_input(
    "Enter your search query",
    placeholder="e.g. Pillars of Creation, purple spiral galaxy, supernova remnant..."
)

# ============================================================
# 点击搜索按钮后执行
# ============================================================
if st.button("Search", type="primary") and query:

    # 显示加载中的提示
    with st.spinner("Searching NASA image archive..."):

        try:
            # ------------------------------------------------
            # 情况一：开启了 RAG（AI 总结）
            # ------------------------------------------------
            if enable_rag:
                response = requests.post(
                    f"{BACKEND_URL}/search/rag",
                    json={
                        "query": query,
                        "top_k": top_k,
                        "mode": current_search_mode
                    },
                    timeout=120     # RAG 需要等 LLM 生成，超时时间设长一点
                )
                data = response.json()

                # 显示 AI 总结
                st.subheader("🤖 AI Image Insights")
                st.info(data.get("ai_overview", "No summary available."))

                # 取出检索结果
                results_meta = data["hybrid_results"]["metadatas"]
                results_ids  = data["hybrid_results"]["ids"]

            # ------------------------------------------------
            # 情况二：普通搜索（不用 RAG）
            # ------------------------------------------------
            else:
                if current_search_mode == "text":
                    endpoint = f"{BACKEND_URL}/search/text"
                else:
                    endpoint = f"{BACKEND_URL}/search/image"

                response = requests.post(
                    endpoint,
                    json={"query": query, "top_k": top_k},
                    timeout=60
                )
                data = response.json()

                # ChromaDB 返回的格式是 data["metadatas"][0]
                results_meta = data["metadatas"][0]
                results_ids  = data["ids"][0]

        except Exception as e:
            st.error(f"Something went wrong: {e}")
            st.stop()

    # ============================================================
    # 展示搜索结果
    # ============================================================
    st.subheader(f"Search Results for: *{query}*")

    # 每行放 3 张图片
    cols = st.columns(3)

    for idx, meta in enumerate(results_meta):
        col = cols[idx % 3]   # 轮流放进三列

        with col:
            # 尝试加载并显示图片
            image_url = meta.get("url", "")
            if image_url:
                try:
                    img_response = requests.get(image_url, timeout=10)
                    img = Image.open(BytesIO(img_response.content))
                    st.image(img, use_column_width=True)
                except Exception:
                    # 图片加载失败就显示占位文字
                    st.markdown("🌌 *Image unavailable*")
            else:
                st.markdown("🌌 *No image URL*")

            # 显示标题
            title = meta.get("title", "Untitled")
            st.markdown(f"**{title}**")

            # 点击展开查看详细描述
            with st.expander("View description"):
                desc = meta.get("desc", meta.get("description", "No description available."))
                st.write(desc)

                # 显示档案 ID
                if idx < len(results_ids):
                    st.caption(f"Archive ID: {results_ids[idx]}")